# Stage 06 — Train Gemma LoRA

| | |
|---|---|
| **Intention** | LoRA fine-tune `google/gemma-2-2b-it` for text → gloss. Requires accepting the HF license + `huggingface-cli login`. |
| **Input** | `data/mbart/{train,dev}.jsonl` |
| **Output** | `artifacts/ckpts/gemma/best/` |
| **Runtime** | ~1–3 h on GPU |


In [1]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))

from ttg.config import DATA_DIR, CHECKPOINTS_DIR, ARTIFACTS, PROJECT_ROOT
print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_DIR     =", DATA_DIR)
print("CHECKPOINTS  =", CHECKPOINTS_DIR)


PROJECT_ROOT = /home/khurshida/Projects/uzsl-text-to-gloss
DATA_DIR     = /home/khurshida/Projects/uzsl-text-to-gloss/data
CHECKPOINTS  = /home/khurshida/Projects/uzsl-text-to-gloss/artifacts/ckpts


In [2]:
SMOKE_TEST = False
BF16 = True
DIRECTION = "text2gloss"  # "text2gloss" or "gloss2text"
EPOCHS = 1 if SMOKE_TEST else 10
BATCH_SIZE = 1
GRAD_ACCUM = 16
LR = 2e-4
MAX_LENGTH = 256
LORA_R, LORA_ALPHA, LORA_DROPOUT = 64, 128, 0.05
SEED = 42
MODEL = "google/gemma-2-2b-it"
OUTPUT_DIR = CHECKPOINTS_DIR / ("gemma_g2t" if DIRECTION == "gloss2text" else "gemma")

In [ ]:
import json
import torch
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from trl import SFTConfig, SFTTrainer
from ttg.config import USER_PROMPT, USER_PROMPT_GLOSS_TO_TEXT
from ttg.data import load_split, swap_direction, to_chat_messages

set_seed(SEED)
train = load_split(DATA_DIR / "mbart" / "train.jsonl")
dev = load_split(DATA_DIR / "mbart" / "dev.jsonl")
if DIRECTION == "gloss2text":
    train, dev = swap_direction(train), swap_direction(dev)
prompt = USER_PROMPT_GLOSS_TO_TEXT if DIRECTION == "gloss2text" else USER_PROMPT

tokenizer = AutoTokenizer.from_pretrained(MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
# Register the fingerspelling marker as a real token so it survives training
# and generation intact instead of being fragmented into subwords.
tokenizer.add_tokens(["[dct]"], special_tokens=False)

# 8-bit base weights: this GPU is shared with other jobs (~17GB already
# resident), and the bf16 base model plus the fp32 cast trl's chunked
# cross-entropy takes of the (vocab_size x hidden) lm_head weight don't fit
# in what's left. Quantizing the frozen base frees ~2.2GB of headroom; LoRA
# + the trainable embed_tokens/lm_head copies stay full precision.
quant_config = BitsAndBytesConfig(load_in_8bit=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=quant_config,
    torch_dtype=torch.bfloat16 if BF16 and torch.cuda.is_available() else torch.float32,
)
model.config.use_cache = False
# Some base checkpoints pad their embedding table beyond the tokenizer's real
# vocab size (e.g. for hardware alignment); only grow, never shrink it, or
# rows for genuine (if unused-by-us) token ids would be silently discarded.
if len(tokenizer) > model.get_input_embeddings().weight.shape[0]:
    model.resize_token_embeddings(len(tokenizer))
# Casts norms to fp32 and enables gradient checkpointing correctly for a
# quantized base (replaces the plain gradient_checkpointing_enable() +
# enable_input_require_grads() calls a non-quantized model would need).
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=LORA_R, lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    # LoRA freezes the base embedding/head; without this the new token's
    # embedding row stays at random init and is never actually learned.
    modules_to_save=["embed_tokens", "lm_head"],
    bias="none",
))
model.print_trainable_parameters()

train_ds = Dataset.from_list(to_chat_messages(train, prompt))
dev_ds = Dataset.from_list(to_chat_messages(dev, prompt))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

trainer = SFTTrainer(
    model=model,
    args=SFTConfig(
        output_dir=str(OUTPUT_DIR),
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR, num_train_epochs=EPOCHS,
        eval_strategy="epoch", save_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model="eval_loss",
        greater_is_better=False, logging_steps=5,
        save_total_limit=1, save_only_model=True,
        seed=SEED, bf16=BF16 and torch.cuda.is_available(),
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        optim="paged_adamw_8bit",
        max_length=MAX_LENGTH, assistant_only_loss=True, report_to=[],
    ),
    train_dataset=train_ds, eval_dataset=dev_ds, processing_class=tokenizer,
)
trainer.train()
best_dir = OUTPUT_DIR / "best"
best_dir.mkdir(parents=True, exist_ok=True)
trainer.model.save_pretrained(str(best_dir))
tokenizer.save_pretrained(str(best_dir))
meta = {"model": MODEL, "direction": DIRECTION, "num_train": len(train.texts), "num_dev": len(dev.texts),
        "epochs": EPOCHS, "batch_size": BATCH_SIZE, "grad_accum": GRAD_ACCUM,
        "lr": LR, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "user_prompt": prompt}
(OUTPUT_DIR / "train_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
print("Saved →", best_dir)
